# 1. Read source data

In [86]:
import sys
import numpy as np
import pandas as pd
from config import PER_ORGANISM_PER_CLASS_ALLELES
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)





In [87]:
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber/"
total_df =  pd.read_csv("{}schreiber_total_data.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))

per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_mt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    per_allele_wt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")
    
def defineOrganism(allele_name):
    for organism_to_use in PER_ORGANISM_PER_CLASS_ALLELES.keys():
        per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use]
        for class_to_use, alleles in per_class_alleles.items():
            if allele_name in alleles:
                return organism_to_use
    print("organism not  found")
    return None


def defineClass(allele_name):
    organism =  defineOrganism(allele_name)
    per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism]
    for class_to_use, alleles in per_class_alleles.items():
        if allele_name in alleles:
            return class_to_use
    print("not found")
    return None

    
ALLELES

['H-2Ld', 'H-2IEd']

# 3. Read Joined predictions

In [88]:

#DATA_PATH_PREDICTIONS = "/home/denklewer/Data/gbm122/" #ubuntu machine
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber/"
# for decoy_key in DECOY_PREFIXES:
per_allele_predictions = dict()
all_methods = set()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df = pd.read_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";")
    per_allele_predictions[allele_name] = total_df
    all_methods.update(list(set([item.split('_')[0] for item in total_df.columns if  item.split('_')[0] not in ['allele', 'peptide']])))


In [89]:
total_df.columns

Index(['allele', 'peptide_mt', 'netMHCIIpan-4.0_score_ba_mt',
       'netMHCIIpan-4.0_score_el_mt', 'netMHCIIpan-4.0_affinity_mt',
       'netMHCIIpan-4.0_percentile_rank_ba_mt',
       'netMHCIIpan-4.0_percentile_rank_el_mt', 'mhcnuggets_score_mt',
       'mhcnuggets_affinity_mt', 'mhcnuggets_percentile_rank_mt',
       'puffin_score_mt', 'puffin_epistemic_var_mt', 'puffin_aleatoric_var_mt',
       'puffin_binding_likelihood_mt', 'puffin_percentile_rank_mt',
       'mhcnuggets_score_advantage', 'mhcnuggets_affinity_advantage',
       'mhcnuggets_percentile_rank_advantage', 'puffin_score_advantage',
       'puffin_epistemic_var_advantage', 'puffin_aleatoric_var_advantage',
       'puffin_binding_likelihood_advantage',
       'puffin_percentile_rank_advantage', 'peptide_wt',
       'netMHCIIpan-4.0_score_ba_wt', 'netMHCIIpan-4.0_score_el_wt',
       'netMHCIIpan-4.0_affinity_wt', 'netMHCIIpan-4.0_percentile_rank_ba_wt',
       'netMHCIIpan-4.0_percentile_rank_el_wt', 'mhcnuggets_score_w

In [90]:
all_methods

{'mhcnuggets',
 'netMHC-3.4',
 'netMHC-4.0',
 'netMHCIIpan-4.0',
 'netMHCpan-4.1',
 'puffin'}

# 4. Apply filtering

In [91]:
def filter_epitopes(df, method_name, advantage = False):
    if method_name == 'netMHC-3.4':
        df = df.loc[df[method_name + "_affinity_mt"] < 500].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif  method_name == 'netMHC-4.0':
        df = df.loc[df[method_name +  "_percentile_rank_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
    elif method_name == 'netMHCpan-4.1':
        df = df.loc[df[method_name + "_percentile_rank_el_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_el_advantage"] > 1].copy()
    elif method_name == 'mhcnuggets':
        df = df.loc[df[method_name + "_percentile_rank_mt"] < 2].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
    elif method_name == 'acme':
        df = df.loc[df[method_name + "_score_mt"] > 0.42].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif method_name == 'mhcseqnet':
        df = df.loc[df[method_name + "_score_mt"] > 0.99].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    elif method_name == 'puffin':
        df = df.loc[df[method_name + "_percentile_rank_mt"] > 0.42].copy()
        if advantage:
            df = df.loc[df[method_name + "_percentile_rank_advantage"] > 1].copy()
#     elif method_name == 'puffin':
#         df = df.loc[df[method_name + "_binding_likelihood_mt"] > 0.5].copy()
#         if advantage:
#             df = df.loc[df[method_name + "_binding_likelihood_advantage"] > 1].copy()
    elif method_name == 'mhcattnnet':
        df = df.loc[df[method_name + "_score_mt"] >= 1].copy()
        if advantage:
            df = df.loc[df[method_name + "_score_advantage"] > 1].copy()
    return df


In [92]:
unsupported_alleles = {method : set() for method in methods}

def add_unsupported_allele(method, alleles):
    methods_set = set(methods)
    if (method in methods_set):
        unsupported_alleles[method].update(alleles)

for allele_name in ALLELES:
    if defineClass(allele_name) == 'II':
        add_unsupported_allele('netMHCpan-4.1', [allele_name])
        add_unsupported_allele('netMHC-3.4', [allele_name])
        add_unsupported_allele('netMHC-4.0', [allele_name])
        add_unsupported_allele('acme', [allele_name])
    else:
        add_unsupported_allele('netMHCIIpan-4.0', [allele_name])
        
    if not defineOrganism(allele_name) == 'Human':
        add_unsupported_allele('mhcseqnet', [allele_name])
        add_unsupported_allele('mhcattnnet', [allele_name])
        add_unsupported_allele('acme', [allele_name])

#special cases
add_unsupported_allele('acme',["HLA-C*04:01", "HLA-C*03:04"])
add_unsupported_allele('puffin',["HLA-C*04:01"])

In [93]:
unsupported_alleles

{'netMHCIIpan-4.0': {'H-2Ld'},
 'netMHC-3.4': {'H-2IEd'},
 'mhcnuggets': set(),
 'netMHCpan-4.1': {'H-2IEd'},
 'netMHC-4.0': {'H-2IEd'},
 'puffin': {'HLA-C*04:01'}}

In [94]:
def per_allele_filter_predictions(per_allele_df, selected_methods, advantage=False):
    per_allele_result = dict()
    for allele_name in ALLELES:
        prediction_df =  per_allele_df[allele_name]
        count = len(prediction_df)
        print(f"{allele_name}:({count}) ->", end=" ")
        for method in selected_methods:
            if allele_name not in unsupported_alleles[method]:
                prediction_df = filter_epitopes(prediction_df, method, advantage)
                count = len(prediction_df)
                print(f"{method}:({count}) ->", end=" ")
        print()
        per_allele_result[allele_name] = prediction_df
    return per_allele_result

def per_allele_add_source_info(per_allele_df, region):
    per_allele_result = dict()
    df_source =  per_region_total_df[region].copy()
    df_source = df_source.rename(columns={ "HLA Allele":"allele", "MT Epitope Seq":"peptide_mt", "WT Epitope Seq":"peptide_wt"}) # rename for convinience
    
    # No Transcript, Protein position , Transcript Expression, Transcript Support Level because it produces many duplicates
    df_source = df_source[
        [ 'allele','peptide_mt', 'peptide_wt','Chromosome', 'Start', 'Stop', 'Reference', 'Variant', 
         'Ensembl Gene ID', 
         'Variant Type','Mutation', 
         'Gene Name', 'HGVSc', 'HGVSp', 'Mutation Position', 
         'Tumor DNA Depth', 'Tumor DNA VAF', 'Tumor RNA Depth', 'Tumor RNA VAF',
         'Normal Depth', 'Normal VAF', 'Gene Expression',
         'cterm_7mer_gravy_score','max_7mer_gravy_score', 'difficult_n_terminal_residue',
       'c_terminal_cysteine', 'c_terminal_proline', 'cysteine_count','n_terminal_asparagine', 'asparagine_proline_bond_count']].drop_duplicates()
    
    
    for allele_name in ALLELES:
        df_top = per_allele_df[allele_name].copy()
        len_initial = len(df_top)
        df_merged = df_source.merge(df_top, on=["allele","peptide_mt", "peptide_wt" ])
        len_merged = len(df_merged)
        df_filtered = df_source[(df_source['peptide_mt'].isin(df_top['peptide_mt'])) & (df_source['peptide_wt'].isin(df_top['peptide_wt'])) & (df_source['allele'].isin(df_top['allele']))]
        len_filtered = len(df_filtered)
        print(f"top epitopes {len_initial}, after merging  {len_merged}, after filtering {len_filtered}")
        if not len_merged == len_initial :
            print(f"many rows for 1 epitope due to multiple Gene ID")
        
        per_allele_result[allele_name] = df_merged
    return per_allele_result

def drop_unsupported_columns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        for method in methods:
            if allele_name in unsupported_alleles[method]:
                columns_to_drop = [col.strip() for col in df.columns if col.startswith(method)]
            
                df = df.drop(columns=columns_to_drop, errors='ignore')    
            per_allele_df[allele_name] = df
        
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df

    
    
# def reorder_result_columns(df_filtered):  
#     df_filtered = df_filtered.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
#      df_filtered = df_filtered.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])

#     return df_filtered


def filter_and_save_to_file(per_region_per_allele_predictions, selected_methods, advantage=False):
    print("1.Filter using methods")
    per_allele_top_epitopes = dict()
    
    per_allele_top_epitopes = per_allele_filter_predictions(per_allele_predictions, selected_methods, advantage=advantage)
    print("2.Drop columns")
    per_allele_top_epitopes = drop_unsupported_columns(per_allele_top_epitopes)
    

    print("2.Write to files")
    #DATA_PATH = "/home/denklewer/Data/gbm122/" #ubuntu machine
    DATA_PATH = PREDICTION_PATH
    for allele_name, df_filtered in per_allele_top_epitopes.items():
        if advantage:
            df_filtered.to_csv("{}top_epitopes_{}_filtered+advantage_{}.csv".format(DATA_PATH, allele_name, "_".join(selected_methods)),sep=";", index=False)
        else:
            df_filtered.to_csv("{}top_epitopes_filtered_all_regions_{}.csv".format(DATA_PATH,allele_name,  "_".join(selected_methods)),sep=";", index=False)


In [95]:
selected_methods = ['netMHCIIpan-4.0', 'netMHCpan-4.1','mhcnuggets','puffin']
dicc  = filter_and_save_to_file(per_allele_predictions, selected_methods, advantage=False)

1.Filter using methods
H-2Ld:(51) -> netMHCpan-4.1:(9) -> mhcnuggets:(5) -> puffin:(2) -> 
H-2IEd:(134) -> netMHCIIpan-4.0:(134) -> mhcnuggets:(50) -> puffin:(50) -> 
2.Drop columns
2.Write to files


In [96]:
selected_methods = ['netMHCIIpan-4.0', 'netMHCpan-4.1','mhcnuggets','puffin']
dicc  = filter_and_save_to_file(per_allele_predictions, selected_methods, advantage=True)

1.Filter using methods
H-2Ld:(51) -> netMHCpan-4.1:(9) -> mhcnuggets:(5) -> puffin:(2) -> 
H-2IEd:(134) -> netMHCIIpan-4.0:(134) -> mhcnuggets:(3) -> puffin:(2) -> 
2.Drop columns
2.Write to files
